# Task 2: Data Preprocessing & Modeling — Uber Fare Prediction
Target: `fare_amount` (regression). All fitted transformers are trained on the train split only.

In [17]:
import pandas as pd

df = pd.read_csv("../data/processed/uber_cleaned_data.csv")
print("shape:", df.shape)
df.info()

print("\nMissing share per column (top 10):")
print(df.isnull().mean().round(4).sort_values(ascending=False).head(10))

print("\nExact duplicates:", df.duplicated().sum())
df.head()

shape: (489273, 23)
<class 'pandas.DataFrame'>
RangeIndex: 489273 entries, 0 to 489272
Data columns (total 23 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   car_condition      489273 non-null  str    
 1   weather            489273 non-null  str    
 2   traffic_condition  489273 non-null  str    
 3   fare_amount        489273 non-null  float64
 4   pickup_longitude   489273 non-null  float64
 5   pickup_latitude    489273 non-null  float64
 6   dropoff_longitude  489273 non-null  float64
 7   dropoff_latitude   489273 non-null  float64
 8   passenger_count    489273 non-null  int64  
 9   hour               489273 non-null  int64  
 10  day                489273 non-null  int64  
 11  month              489273 non-null  int64  
 12  weekday            489273 non-null  int64  
 13  year               489273 non-null  int64  
 14  jfk_dist           489273 non-null  float64
 15  ewr_dist           489273 non-null  float6

,car_condition,weather,traffic_condition,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,hour,...,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,bearing,road_distance_km,bearing_deg,haversine_dist_km
0,Very Good,windy,Congested Traffic,4.5,-1.288826,0.710721,-1.288779,0.710563,1,17,...,2009,20.265840,55.176046,14.342611,34.543548,27.572573,-2.918897,1.2445,192.759531,1.030765
1,Excellent,cloudy,Flow Traffic,16.9,-1.291824,0.710546,-1.291182,0.711780,1,16,...,2010,44.667679,31.832358,23.130775,15.125872,8.755732,-0.375217,9.4069,338.501634,8.450145
2,Bad,stormy,Congested Traffic,5.7,-1.291242,0.711418,-1.291391,0.711231,2,0,...,2011,43.597686,33.712082,19.865289,17.722624,9.847344,2.599961,1.3877,148.966789,1.389527
3,Very Good,stormy,Flow Traffic,7.7,-1.291319,0.710927,-1.291396,0.711363,1,4,...,2012,42.642965,32.556289,21.063132,15.738963,7.703421,0.133905,3.8888,7.672164,2.799274
4,Bad,stormy,Congested Traffic,5.3,-1.290987,0.711536,-1.290787,0.711811,1,7,...,2010,43.329953,39.406828,15.219339,23.732406,15.600745,-0.502703,1.9986,331.197217,1.999160


In [18]:
# Old OSRM run (no radiuses): ~202 rows with road = 0 but straight line > 100 m. Fixed run: 1 row.
zero_far = (df["road_distance_km"] == 0) & (df["haversine_dist_km"] > 0.1)
print("road = 0 but haversine > 100 m:", int(zero_far.sum()))

# Snapped-to-a-far-road signature: tiny straight line, very long road
long_road = (df["haversine_dist_km"] < 0.5) & (df["road_distance_km"] > 3)
print("haversine < 0.5 km but road > 3 km:", int(long_road.sum()))

df[zero_far | long_road][["haversine_dist_km", "road_distance_km", "fare_amount"]].head(10)

road = 0 but haversine > 100 m: 0
haversine < 0.5 km but road > 3 km: 158


,haversine_dist_km,road_distance_km,fare_amount
7733,0.105735,5.2269,2.5
9293,0.325908,5.0414,3.0
16919,0.156852,3.1169,2.9
18724,0.001122,4.1042,45.0
19435,0.041698,3.3965,2.5
24355,0.428673,5.9253,9.0
29982,0.371237,6.2151,6.1
32503,0.419759,4.9916,6.5
37722,0.242397,4.2703,4.1
49466,0.097344,7.6564,2.5


In [19]:
pd.set_option("display.width", 200)

print(df.describe().T[["min", "50%", "max"]].round(3))

for c in ["car_condition", "weather", "traffic_condition"]:
    print(f"\n{c}:")
    print(df[c].value_counts())

                        min       50%       max
fare_amount           2.500     8.500   250.000
pickup_longitude     -1.300    -1.291    -1.272
pickup_latitude       0.705     0.711     0.719
dropoff_longitude    -1.300    -1.291    -1.272
dropoff_latitude      0.705     0.711     0.719
passenger_count       1.000     1.000     6.000
hour                  0.000    14.000    23.000
day                   1.000    16.000    31.000
month                 1.000     6.000    12.000
weekday               0.000     3.000     6.000
year               2009.000  2012.000  2015.000
jfk_dist              1.018    42.479   156.212
ewr_dist              1.461    34.667   222.699
lga_dist              0.382    19.494   176.480
sol_dist              0.533    18.202   201.802
nyc_dist              0.080    10.323   196.085
bearing              -3.142    -0.100     3.142
road_distance_km      0.000     2.843   120.860
bearing_deg           0.000   186.370   359.999
haversine_dist_km     0.000     2.154   

## Train/Test Split
Random 80/20 split: each row is an independent trip and there is no entity ID or full timestamp, so neither a chronological split nor GroupKFold is needed. It is done before any imputer/scaler/encoder is fitted to avoid leakage.

In [20]:
from sklearn.model_selection import train_test_split

TARGET = "fare_amount"
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print(pd.DataFrame({"train": y_train.describe(), "test": y_test.describe()}).round(2))
print("\nTarget skewness (train):", round(y_train.skew(), 2))

X_train: (391418, 22) | X_test: (97855, 22)
           train      test
count  391418.00  97855.00
mean       11.33     11.34
std         9.66      9.66
min         2.50      2.50
25%         6.00      6.00
50%         8.50      8.50
75%        12.50     12.50
max       250.00    215.00

Target skewness (train): 3.61


## Feature Engineering
- Coordinates are stored in radians; convert to degrees for interpretability (they stay numeric either way, so this doesn't affect the model, only readability during EDA/debugging).
- `hour` is cyclical (23 and 0 are adjacent), so encode it as sin/cos instead of leaving it as a raw integer or one-hot (24 columns).
- `bearing_deg` is also cyclical (0° and 360° are the same direction) — same treatment.
- Drop `bearing` (radians) and raw `hour`/`bearing_deg` once their cyclical versions exist, to avoid redundant/duplicate information.

In [21]:
import numpy as np

def engineer(X):
    X = X.copy()

    coord_cols = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"]
    X[coord_cols] = np.degrees(X[coord_cols])

    X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

    X["bearing_sin"] = np.sin(np.radians(X["bearing_deg"]))
    X["bearing_cos"] = np.cos(np.radians(X["bearing_deg"]))

    X["is_weekend"] = X["weekday"].isin([5, 6]).astype(int)

    return X.drop(columns=["hour", "bearing", "bearing_deg"])

X_train = engineer(X_train)
X_test = engineer(X_test)

print(X_train.shape, X_test.shape)
X_train.filter(regex="hour|bearing|weekend|longitude|latitude").head()

(391418, 24) (97855, 24)


,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,hour_sin,hour_cos,bearing_sin,bearing_cos,is_weekend
317596,-73.991769,40.750249,-73.996759,40.767496,0.258819,-9.659258e-01,0.214045,0.976824,1
458053,-73.964853,40.806905,-73.973718,40.796322,-1.000000,-1.836970e-16,0.535552,-0.844502,0
62696,-73.983560,40.748165,-73.872974,40.774312,-0.707107,-7.071068e-01,-0.954385,0.298579,0
282697,-73.959905,40.710883,-73.944563,40.714493,0.707107,7.071068e-01,-0.955016,0.296556,1
466866,-73.953517,40.782260,-73.961297,40.772577,-0.965926,2.588190e-01,0.519819,-0.854276,0


## Outlier Detection & Treatment
- `road_distance_km` / `haversine_dist_km`: cap at the 99th percentile of the train set (real long trips are valid, not errors — capping preserves them instead of deleting).
- Rows where `haversine_dist_km < 0.5` and `road_distance_km > 3`: earlier investigation (see Task 1 follow-up) showed these charge like short trips, meaning the road distance is unreliable, not a rare-but-real long detour. Flag with an indicator column rather than dropping, so the model can learn to distrust it, and impute the value itself with the median.
- `fare_amount`: cap at the 99th percentile of the train set, computed on the log scale to avoid a few extreme fares distorting the bound.

In [22]:
road_cap = X_train["road_distance_km"].quantile(0.99)
hav_cap = X_train["haversine_dist_km"].quantile(0.99)
fare_cap = np.expm1(np.log1p(y_train).quantile(0.99))

print("road_distance_km cap :", round(road_cap, 2), "km")
print("haversine_dist_km cap:", round(hav_cap, 2), "km")
print("fare_amount cap      :", round(fare_cap, 2))

bad_road_train = (X_train["haversine_dist_km"] < 0.5) & (X_train["road_distance_km"] > 3)
bad_road_test = (X_test["haversine_dist_km"] < 0.5) & (X_test["road_distance_km"] > 3)
print("\nFlagged unreliable-road rows -> train:", int(bad_road_train.sum()), "| test:", int(bad_road_test.sum()))

print("\nRows above fare cap -> train:", int((y_train > fare_cap).sum()), "| test:", int((y_test > fare_cap).sum()))

road_distance_km cap : 27.86 km
haversine_dist_km cap: 20.3 km
fare_amount cap      : 52.0

Flagged unreliable-road rows -> train: 134 | test: 24

Rows above fare cap -> train: 3843 | test: 968


In [23]:
#Apply the outlier treatment (bounds computed on train, applied to both splits)
def treat_outliers(X, road_cap, hav_cap):
    X = X.copy()
    bad_road = (X["haversine_dist_km"] < 0.5) & (X["road_distance_km"] > 3)
    X["road_distance_unreliable"] = bad_road.astype(int)
    X.loc[bad_road, "road_distance_km"] = X_train.loc[~bad_road_train, "road_distance_km"].median()

    X["road_distance_km"] = X["road_distance_km"].clip(upper=road_cap)
    X["haversine_dist_km"] = X["haversine_dist_km"].clip(upper=hav_cap)
    return X

X_train = treat_outliers(X_train, road_cap, hav_cap)
X_test = treat_outliers(X_test, road_cap, hav_cap)

y_train_c = y_train.clip(upper=fare_cap)
y_test_c = y_test.clip(upper=fare_cap)

print(X_train[["road_distance_km", "haversine_dist_km", "road_distance_unreliable"]].describe().round(2))
print("\ny_train max after cap:", y_train_c.max(), "| y_test max after cap:", y_test_c.max())

       road_distance_km  haversine_dist_km  road_distance_unreliable
count         391418.00          391418.00                 391418.00
mean               4.42               3.30                      0.00
std                4.74               3.49                      0.02
min                0.00               0.00                      0.00
25%                1.71               1.25                      0.00
50%                2.84               2.15                      0.00
75%                5.09               3.92                      0.00
max               27.86              20.30                      1.00

y_train max after cap: 52.00000000000001 | y_test max after cap: 52.00000000000001


## Encoding & Scaling
- `car_condition` and `traffic_condition` have a natural order (Bad < Good < Very Good < Excellent; Flow < Dense < Congested) -> OrdinalEncoder with an explicit category order.
- `weather` is nominal with 5 low-cardinality levels -> OneHotEncoder.
- All other numeric features -> StandardScaler, since the baseline model is a linear one (Ridge) which is scale-sensitive. `_sin`/`_cos`/`is_weekend`/`road_distance_unreliable` columns are already bounded/binary, so scaling them is harmless but unnecessary; they are scaled anyway for simplicity since it doesn't hurt a linear model.
- Everything is wrapped in a single ColumnTransformer so the same fitted transformer applies identically to train and test, and can be reused inside cross-validation without leakage.

In [24]:
#Build (but don't yet fit) the ColumnTransformer for encoding + scaling
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ordinal_cols = ["car_condition", "traffic_condition"]
ordinal_categories = [
    ["Bad", "Good", "Very Good", "Excellent"],
    ["Flow Traffic", "Dense Traffic", "Congested Traffic"],
]
onehot_cols = ["weather"]
numeric_cols = [c for c in X_train.columns if c not in ordinal_cols + onehot_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), onehot_cols),
        ("num", StandardScaler(), numeric_cols),
    ]
)

print("ordinal :", ordinal_cols)
print("onehot  :", onehot_cols)
print("numeric :", len(numeric_cols), "columns")
numeric_cols

ordinal : ['car_condition', 'traffic_condition']
onehot  : ['weather']
numeric : 22 columns


['pickup_longitude',
 'pickup_latitude',
 'dropoff_longitude',
 'dropoff_latitude',
 'passenger_count',
 'day',
 'month',
 'weekday',
 'year',
 'jfk_dist',
 'ewr_dist',
 'lga_dist',
 'sol_dist',
 'nyc_dist',
 'road_distance_km',
 'haversine_dist_km',
 'hour_sin',
 'hour_cos',
 'bearing_sin',
 'bearing_cos',
 'is_weekend',
 'road_distance_unreliable']

## Feature Selection
- First pass: correlation matrix on numeric train features to spot near-duplicate/redundant pairs (e.g. `weekday` vs `is_weekend`, or the 5 distance-to-landmark columns vs `road_distance_km`/`haversine_dist_km`) before touching any model.
- Embedded pass: Lasso coefficients on the encoded+scaled train set, to see which features shrink to ~0.

In [25]:
#Correlation matrix on the raw numeric train features (before encoding/scaling)
corr = X_train[numeric_cols].corr()

# Print pairs with |correlation| > 0.85 (excluding the diagonal)
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .rename("corr")
    .reset_index()
)
pairs.columns = ["feature_1", "feature_2", "corr"]
high = pairs[pairs["corr"].abs() > 0.85].sort_values("corr", key=abs, ascending=False)
print(high.to_string(index=False))

       feature_1         feature_2     corr
road_distance_km haversine_dist_km 0.986518
        sol_dist          nyc_dist 0.984068
        ewr_dist          sol_dist 0.954524
        ewr_dist          nyc_dist 0.939043


In [26]:
#Drop redundant features identified by the correlation matrix, then check remaining top correlations
drop_cols = ["haversine_dist_km", "sol_dist", "ewr_dist"]

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)
numeric_cols = [c for c in numeric_cols if c not in drop_cols]

corr2 = X_train[numeric_cols].corr()
pairs2 = (
    corr2.where(np.triu(np.ones(corr2.shape), k=1).astype(bool))
    .stack()
    .rename("corr")
    .reset_index()
)
pairs2.columns = ["feature_1", "feature_2", "corr"]
print(pairs2[pairs2["corr"].abs() > 0.85].sort_values("corr", key=abs, ascending=False).to_string(index=False))

print("\nremaining numeric columns:", len(numeric_cols))

Empty DataFrame
Columns: [feature_1, feature_2, corr]
Index: []

remaining numeric columns: 19


In [27]:
#Rebuild the ColumnTransformer with the reduced numeric columns, then check Lasso coefficients
preprocessor = ColumnTransformer(
    transformers=[
        ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), onehot_cols),
        ("num", StandardScaler(), numeric_cols),
    ]
)

from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

lasso_probe = Pipeline([
    ("prep", preprocessor),
    ("model", Lasso(alpha=0.01, random_state=42)),
])
lasso_probe.fit(X_train, np.log1p(y_train_c))

feature_names = lasso_probe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lasso_probe.named_steps["model"].coef_, index=feature_names).sort_values(key=abs, ascending=False)

print("Non-zero features:", (coefs != 0).sum(), "/", len(coefs))
print("\nTop 10 by |coefficient|:")
print(coefs.head(10).round(4))
print("\nFeatures shrunk to exactly zero:")
print(coefs[coefs == 0])

Non-zero features: 9 / 26

Top 10 by |coefficient|:
num__road_distance_km    0.4480
num__year                0.0572
num__nyc_dist           -0.0080
num__month               0.0075
num__hour_cos           -0.0068
num__pickup_longitude   -0.0067
num__jfk_dist            0.0066
num__lga_dist           -0.0025
num__dropoff_latitude   -0.0001
ord__car_condition       0.0000
dtype: float64

Features shrunk to exactly zero:
ord__car_condition               0.0
num__dropoff_longitude          -0.0
num__pickup_latitude             0.0
ohe__weather_windy               0.0
ohe__weather_sunny              -0.0
ohe__weather_rainy               0.0
ohe__weather_stormy             -0.0
ohe__weather_cloudy              0.0
ord__traffic_condition          -0.0
num__weekday                    -0.0
num__day                         0.0
num__passenger_count             0.0
num__hour_sin                   -0.0
num__bearing_sin                 0.0
num__bearing_cos                -0.0
num__is_weekend         

## Baseline Model (Ridge, log-target, with embedded feature selection)
- Target is log1p-transformed inside `TransformedTargetRegressor` (skewness = 3.61 from Cell 3), predictions are inverse-transformed automatically before scoring.
- Feature selection (`SelectFromModel` with a Lasso estimator) is embedded inside the pipeline rather than applied once on the full train set, so each CV fold selects its own features and the CV score stays an honest estimate of generalization.
- 5-fold KFold CV (data is i.i.d.: no time order or shared-entity grouping was found), scored on RMSE.

In [28]:
#Baseline pipeline — Ridge with log-target and embedded Lasso-based feature selection
from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate

base_pipeline = Pipeline([
    ("prep", preprocessor),
    ("select", SelectFromModel(Lasso(alpha=0.01, random_state=42))),
    ("model", TransformedTargetRegressor(
        regressor=Ridge(alpha=1.0, random_state=42),
        func=np.log1p, inverse_func=np.expm1,
    )),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    base_pipeline, X_train, y_train_c, cv=cv,
    scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"],
)

print("RMSE:", (-scores["test_neg_root_mean_squared_error"]).mean().round(3),
      "±", (-scores["test_neg_root_mean_squared_error"]).std().round(3))
print("MAE :", (-scores["test_neg_mean_absolute_error"]).mean().round(3))
print("R2  :", scores["test_r2"].mean().round(3))

RMSE: 6.009 ± 0.081
MAE : 2.786
R2  : 0.548


## Hyperparameter-Tuned Model (Gradient Boosting)
- Ridge assumes a linear relationship; the Lasso probe showed the signal is dominated by one feature, suggesting the linear model can't capture interactions (e.g. distance × traffic, time-of-day surge). A tree-based model captures these without manual polynomial features.
- `HistGradientBoostingRegressor` needs no scaling and is robust to the remaining redundant features, so feature selection is skipped for this pipeline; encoding is still needed for the categorical columns.
- Tuned with `RandomizedSearchCV` (large-ish search space, tree ensembles are relatively cheap to fit) using the same 5-fold CV scheme as the baseline, scored on RMSE, with a locked-away test set never touched here.

In [29]:
# Hyperparameter-tuned Gradient Boosting model
from scipy.stats import randint, uniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

gb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(random_state=42),
        func=np.log1p, inverse_func=np.expm1,
    )),
])

param_dist = {
    "model__regressor__max_iter": randint(100, 400),
    "model__regressor__max_depth": randint(3, 12),
    "model__regressor__learning_rate": uniform(0.03, 0.27),
    "model__regressor__l2_regularization": uniform(0.0, 1.0),
}

search = RandomizedSearchCV(
    gb_pipeline, param_dist, n_iter=25, cv=cv,
    scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1,
)
search.fit(X_train, y_train_c)

print("Best params:", search.best_params_)
print("Best CV RMSE:", round(-search.best_score_, 3))

Best params: {'model__regressor__l2_regularization': np.float64(0.230893825622149), 'model__regressor__learning_rate': np.float64(0.09507687582702316), 'model__regressor__max_depth': 9, 'model__regressor__max_iter': 363}
Best CV RMSE: 3.269


## Final Evaluation on the Held-Out Test Set
Both models (baseline Ridge, tuned Gradient Boosting) are refit on the full train set and evaluated once on the untouched test set. This is the only time the test set is used.

In [30]:
#Fit both final models on the full train set, evaluate once on the test set
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base_pipeline.fit(X_train, y_train_c)
best_gb_pipeline = search.best_estimator_  # already refit on full X_train by RandomizedSearchCV (refit=True default)

results = []
for name, pipe in [("Baseline (Ridge)", base_pipeline), ("Tuned (Gradient Boosting)", best_gb_pipeline)]:
    pred = pipe.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test_c, pred),
        "RMSE": mean_squared_error(y_test_c, pred) ** 0.5,
        "R2": r2_score(y_test_c, pred),
    })

results_df = pd.DataFrame(results).round(3)
results_df

,model,MAE,RMSE,R2
0,Baseline (Ridge),2.748,5.903,0.566
1,Tuned (Gradient Boosting),1.499,3.217,0.871


## Save the Final Pipeline & Summary
The tuned Gradient Boosting pipeline is saved as the production model (best MAE/RMSE/R2 on the untouched test set).

In [31]:
#Persist the final pipeline and print the summary deliverable
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
joblib.dump(best_gb_pipeline, "../models/final_pipeline.joblib")

X_train.to_csv("../data/processed/train_processed.csv", index=False)
X_test.to_csv("../data/processed/test_processed.csv", index=False)
y_train_c.to_csv("../data/processed/y_train.csv", index=False)
y_test_c.to_csv("../data/processed/y_test.csv", index=False)

print("Saved: ../models/final_pipeline.joblib")
print("Saved processed train/test feature and target CSVs to data/processed/")
results_df

Saved: ../models/final_pipeline.joblib
Saved processed train/test feature and target CSVs to data/processed/


,model,MAE,RMSE,R2
0,Baseline (Ridge),2.748,5.903,0.566
1,Tuned (Gradient Boosting),1.499,3.217,0.871


## What I Would Try Next
- **Target/mean encoding** on a re-added high-cardinality location feature (e.g. a coarse geo-grid of pickup/dropoff cells) instead of relying only on distance-to-landmark features — pickup/dropoff *area* likely carries pricing signal (e.g. airport zones, surge-prone districts) beyond straight-line distance.
- **XGBoost/LightGBM with early stopping** instead of `HistGradientBoostingRegressor`, and a wider/Bayesian search (Optuna) over more trials — the tuned model here used only 25 RandomizedSearchCV iterations, so there is likely still room to improve.
- **Revisit `road_distance_km` for the 158 flagged "unreliable" rows**: instead of median imputation, try re-deriving distance from Haversine × a fitted road/straight-line ratio, since Haversine is available and correlated (0.987) with the reliable rows.
- **Investigate whether `car_condition`/`weather`/`traffic_condition` are synthetic**: their near-uniform level distributions and zero Lasso coefficients suggest they may carry no real signal; confirm with the Gradient Boosting model's `feature_importances_` before dropping them from a future iteration.
- **SHAP values** on the tuned Gradient Boosting model to explain individual predictions to non-technical stakeholders, beyond the global MAE/RMSE/R2 table.